In [ ]:
import numpy as npimport pandas as pdimport statsmodels.api as smimport statsmodels.graphics.tsaplots as tsaplotsfrom statsmodels.stats.diagnostic import acorr_breusch_godfreyfrom statsmodels.regression.linear_model import GLS, GLSARfrom datetime import datetimefrom pandas_datareader import data as webimport matplotlib.pyplot as pltfrom statsmodels.tsa.seasonal import seasonal_decomposedef get_fred_data(series_id, start_date="2000-01-01", end_date=None):    if end_date is None:        end_date = datetime.now().strftime("%Y-%m-%d")    return web.DataReader(series_id, 'fred', start_date, end_date).dropna()series_id = "MICH"mich_data = get_fred_data(series_id)mich_data = mich_data.pct_change().dropna()mich_data = mich_data.rename(columns={series_id: "MICH"})for lag in range(1, 3):    mich_data[f"MICH_lag{lag}"] = mich_data["MICH"].shift(lag)mich_data.dropna(inplace=True)X_lags = ["MICH", "MICH_lag1", "MICH_lag2"]X_matrix = sm.add_constant(mich_data[X_lags])y_vector = mich_data["MICH"]model = sm.OLS(y_vector, X_matrix).fit()bg_test = acorr_breusch_godfrey(model, nlags=2)print(f"Breusch-Godfrey p-value: {bg_test[1]:.4f}")gls_model = GLS(y_vector, X_matrix).fit()cochrane_orcutt = GLSAR(y_vector, X_matrix, rho=1).iterative_fit()model_robust = model.get_robustcov_results(cov_type="HAC", maxlags=2)residuals = model.residfig, axes = plt.subplots(2, 1, figsize=(12, 8))tsaplots.plot_acf(residuals, lags=20, alpha=0.05, ax=axes[0])axes[0].set_xlabel("Lag")axes[0].set_ylabel("Autocorrelation")axes[0].spines['top'].set_visible(False)axes[0].spines['right'].set_visible(False)decomposition = seasonal_decompose(mich_data["MICH"], model="additive", period=10)axes[1].plot(mich_data.index, mich_data["MICH"], 'k-', linewidth=1, alpha=0.6, label='Original')axes[1].plot(mich_data.index, decomposition.trend, 'b-', linewidth=1.5, label='Trend')axes[1].set_xlabel("Date")axes[1].set_ylabel("Value")axes[1].legend(frameon=False, loc='best')axes[1].spines['top'].set_visible(False)axes[1].spines['right'].set_visible(False)plt.tight_layout()plt.show()"""initial version"""import numpy as npimport pandas as pdimport statsmodels.api as smfrom statsmodels.stats.diagnostic import acorr_breusch_godfrey# Set seed for reproducibilitynp.random.seed(42)# Generate independent variable (advertising spend)n = 100X = np.random.rand(n) * 100  # Advertising spend in $1000s# Generate serially correlated errors using an AR(1) processrho = 0.6  # Level of serial correlationerrors = np.zeros(n)errors[0] = np.random.randn()for t in range(1, n):    errors[t] = rho * errors[t - 1] + np.random.randn()# Generate dependent variable (sales) with lag effects and correlated errorsbeta = [0.5, 0.3, 0.1]Y = np.zeros(n)for t in range(2, n):    Y[t] = beta[0] * X[t] + beta[1] * X[t-1] + beta[2] * X[t-2] + errors[t]# Convert to DataFramedata = pd.DataFrame({"Y": Y, "X": X})for lag in range(1, 3):    data[f"X_lag{lag}"] = data["X"].shift(lag)# Drop missing values due to laggingdata.dropna(inplace=True)# Fit a distributed lag modelX_lags = ["X", "X_lag1", "X_lag2"]X_matrix = sm.add_constant(data[X_lags])y_vector = data["Y"]model = sm.OLS(y_vector, X_matrix).fit()# Perform the Breusch-Godfrey test for serial correlationbg_test = acorr_breusch_godfrey(model, nlags=2)print(f"Breusch-Godfrey Test p-value: {bg_test[1]:.4f}")# If p-value < 0.05, serial correlation is present.If the p-value < 0.05, we reject the null hypothesis of no serial correlation, indicating that our model suffers from autocorrelation.4. Addressing Serial CorrelationIf serial correlation is detected, there are several ways to correct it:1. Generalized Least Squares (GLS)GLS modifies OLS by accounting for the structure of the serial correlation:from statsmodels.regression.linear_model import GLSgls_model = GLS(y_vector, X_matrix).fit()print(gls_model.summary())2. Cochrane-Orcutt MethodThis iterative procedure transforms the regression model to eliminate serial correlation.from statsmodels.regression.linear_model import GLSARcochrane_orcutt = GLSAR(y_vector, X_matrix, rho=1).iterative_fit()print(cochrane_orcutt.summary())3. Newey-West Standard ErrorsIf correcting the model structure is not feasible, robust standard errors (Newey-West) provide valid inference.model_robust = model.get_robustcov_results(cov_type="HAC", maxlags=2)print(model_robust.summary())5. Visualizing Serial CorrelationTo diagnose serial correlation, we can plot the Autocorrelation Function (ACF) of the residuals.import statsmodels.graphics.tsaplots as tsaplots# Extract residualsresiduals = model.resid# Plot ACFplt.figure(figsize=(10, 5))tsaplots.plot_acf(residuals, lags=20, alpha=0.05)plt.xlabel("Lag")plt.ylabel("Autocorrelation")plt.title("Autocorrelation of Residuals")plt.savefig("/mnt/data/residual_acf.png")plt.show()